In [ ]:
import pandas as pd

# Đọc dữ liệu mới từ file Excel
data = pd.read_excel("my_reviews.xlsx")
print(f"Kích thước dữ liệu: {data.shape}")
print("\nCác cột trong data:")
print(data.columns.tolist())
print("\nMột số dòng đầu tiên:")
data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'my_reviews.csv'

In [ ]:
# Kiểm tra dữ liệu
print("Thông tin về dữ liệu:")
print(f"Tổng số reviews: {len(data)}")
print(f"Số công ty: {data['company_name'].nunique()}")
print(f"\nKiểm tra giá trị null:")
print(data.isnull().sum())
print(f"\nMột số review mẫu:")
data[["company_name", "review_content"]].head()

Các cột trong data:
['Company Name', 'Cmt_day', 'Title', 'What I liked', 'Suggestions for improvement', 'Rating', 'Salary & benefits', 'Training & learning', 'Management cares about me', 'Culture & fun', 'Office & workspace', 'Recommend?']

Kiểm tra một số dòng đầu tiên:


,Company Name,Title,What I liked,Suggestions for improvement
0,Success Software Services,Phúc lợi khá OK còn công việc thì tùy dự án và...,"Văn phòng khá đẹp, nhiều event trong nội bộ cô...",Nên cải thiện quy trình và phúc lợi dành cho o...
1,Success Software Services,"Môi trường năng động, trẻ, chính sách bảo mật ...","Môi trường năng động, trẻ trung.\nCó cơ hội là...",Tăng số ngày nghỉ phép như 12 phép năm + 3 phé...
2,Success Software Services,Một nơi tốt để học và phát triểu bản thân,"Định hướng công ty tốt, khách hàng châu mỹ nên...","Cần đầu tư hơn về cơ sở vật chất, văn phòng , ..."
3,Success Software Services,"Review mang tính góp ý, không có ý định chê cô...","Đồng nghiệp thân thiện, hoà đồng, hỗ trợ hết m...","- Ràng buộc thực tập sinh 2 năm, nhưng lúc phỏ..."
4,Success Software Services,"Cty trẻ, năng động, lead có tâm, benefit ít.","Cty trẻ, năng động, lead có tâm. Process khá c...",Nên thêm các benefit tương ứng khi member làm ...


In [ ]:
import re

def clean_review(text):
    if not isinstance(text, str) or not text:
        return ""

    # BƯỚC 1: Xử lý các biến thể của xuống dòng để split chính xác
    text = text.replace('\\n', '\n')
    
    # BƯỚC 2: Xử lý Emoji/Emoticons trước khi chúng bị băm nhỏ bởi regex ký tự đặc biệt
    text = re.sub(r':[Dd]+\b', ' ', text)
    text = re.sub(r':\)+', ' ', text)  # Xóa :), :)), :))), v.v.
    text = re.sub(r'(😕)+', ' ', text)
    text = re.sub(r':\(+', ' ', text)
    text = re.sub(r'(😊)+', ' ', text)
    text = re.sub(r'>[-_.]<', ' ', text)  # Xóa >_<, >.<, >-<
    # Xử lý các emoticon dạng (^-^), (^_^), ^.^, v.v.
    text = re.sub(r'\(\^[-_]\^\)', ' ', text)
    text = re.sub(r'\^[._-]?\^', ' ', text)
    text = re.sub(r'\(\^o?\^\)', ' ', text)

    # BƯỚC 3: Tách dữ liệu thành từng dòng (List of lines)
    lines = text.split('\n')
    cleaned_lines = []

    # BƯỚC 4: Duyệt qua từng dòng để xóa sạch dấu gạch, số, sao...
    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Xóa # ở đầu dòng
        line = re.sub(r'^#\s*', '', line)
        
        # Xóa các ký tự đầu dòng không mong muốn
        line = re.sub(r'^[ \t\-\*\+\•\d\.\)\(\[\]]+', '', line)
        
        # Xóa các ký tự đặc biệt cuối dòng như ^, *, #
        line = re.sub(r'[\^\*#]+\s*$', '', line)
        
        # Xóa các dấu chấm liên tiếp
        line = re.sub(r'\.{2,}', '', line)
        
        # Xóa dấu phẩy cuối câu
        line = re.sub(r',\s*$', '', line)
        
        # Dọn dẹp khoảng trắng
        line = re.sub(r'\s+', ' ', line).strip()

        if line:
            # Thêm dấu chấm nếu chưa có dấu câu cuối
            if not re.search(r'[.!?]$', line):
                line += '.'
            cleaned_lines.append(line)

    # BƯỚC 5: Loại bỏ duplicate sentences (giữ thứ tự, không phân biệt hoa thường)
    seen = set()
    unique_lines = []
    for line in cleaned_lines:
        key = line.lower().strip()
        if key not in seen:
            seen.add(key)
            unique_lines.append(line)
    
    result = ' '.join(unique_lines)
    result = re.sub(r'\.+', '.', result)
    result = re.sub(r'\s\.', '.', result)
    result = re.sub(r'\s+', ' ', result)
    
    return result.strip()


def mask_company_name_in_text(text, company_name):
    """
    Mask tên công ty thành [COMPANY] trong text
    """
    if not isinstance(text, str) or not text or not isinstance(company_name, str):
        return text
    
    # Mask tên công ty (case insensitive) với word boundaries
    company_escaped = re.escape(company_name)
    pattern = r'\b' + company_escaped + r'\b'
    result = re.sub(pattern, '[COMPANY]', text, flags=re.IGNORECASE)
    
    # Dọn dẹp khoảng trắng thừa
    result = re.sub(r'\s+', ' ', result).strip()
    
    return result


In [ ]:
# Làm sạch nội dung review
print("Đang làm sạch nội dung review...")
data["review_content_cleaned"] = data["review_content"].astype("str").apply(clean_review)
print("Đã làm sạch xong!")
print(f"\nĐộ dài trung bình review sau khi clean: {data['review_content_cleaned'].str.len().mean():.0f} ký tự")
data[["review_content", "review_content_cleaned"]].head()

,Company Name,Cmt_day,Title,What I liked,Suggestions for improvement,Rating,Salary & benefits,Training & learning,Management cares about me,Culture & fun,Office & workspace,Recommend?
0,Success Software Services,2024-06-01,Phúc lợi khá OK còn công việc thì tùy dự án và...,"Văn phòng khá đẹp, nhiều event trong nội bộ cô...",Nên cải thiện quy trình và phúc lợi dành cho o...,3,3,3,3,3,3,Yes
1,Success Software Services,2023-09-01,"Môi trường năng động, trẻ, chính sách bảo mật ...","Môi trường năng động, trẻ trung. Có cơ hội làm...",Tăng số ngày nghỉ phép như 12 phép năm + 3 phé...,4,4,4,3,3,4,Yes
2,Success Software Services,2021-07-01,Một nơi tốt để học và phát triểu bản thân.,"Định hướng công ty tốt, khách hàng châu mỹ nên...","Cần đầu tư hơn về cơ sở vật chất, văn phòng , ...",5,4,5,5,4,3,Yes
3,Success Software Services,2024-03-01,"Review mang tính góp ý, không có ý định chê cô...","Đồng nghiệp thân thiện, hoà đồng, hỗ trợ hết m...","Ràng buộc thực tập sinh 2 năm, nhưng lúc phỏng...",4,2,2,4,4,3,No
4,Success Software Services,2024-02-01,"Cty trẻ, năng động, lead có tâm, benefit ít.","Cty trẻ, năng động, lead có tâm. Process khá c...",Nên thêm các benefit tương ứng khi member làm ...,3,2,3,3,2,4,No


In [ ]:
# Mask tên công ty trong review content
print("Đang mask tên công ty từ review...")

# Hàm xử lý mask cho từng dòng
def mask_company_in_review(row):
    review = row["review_content_cleaned"]
    company = row["company_name"]
    # Lấy tên công ty (bỏ phần số lượng review trong ngoặc nếu có)
    import re
    company_clean = re.sub(r'\s*\(\d+\)$', '', company) if isinstance(company, str) else company
    return mask_company_name_in_text(review, company_clean)

data["review_content_masked"] = data.apply(mask_company_in_review, axis=1)

print("Đã mask tên công ty xong!")
print("\nKiểm tra một vài dòng sau khi mask:")
data[["company_name", "review_content_masked"]].head(10)

Đang mask tên công ty từ các cột review...
Đã mask tên công ty xong!

Kiểm tra một vài dòng sau khi mask:


,Company Name,Title,What I liked,Suggestions for improvement
0,Success Software Services,Phúc lợi khá OK còn công việc thì tùy dự án và...,"Văn phòng khá đẹp, nhiều event trong nội bộ cô...",Nên cải thiện quy trình và phúc lợi dành cho o...
1,Success Software Services,"Môi trường năng động, trẻ, chính sách bảo mật ...","Môi trường năng động, trẻ trung. Có cơ hội làm...",Tăng số ngày nghỉ phép như 12 phép năm + 3 phé...
2,Success Software Services,Một nơi tốt để học và phát triểu bản thân.,"Định hướng công ty tốt, khách hàng châu mỹ nên...","Cần đầu tư hơn về cơ sở vật chất, văn phòng , ..."
3,Success Software Services,"Review mang tính góp ý, không có ý định chê cô...","Đồng nghiệp thân thiện, hoà đồng, hỗ trợ hết m...","Ràng buộc thực tập sinh 2 năm, nhưng lúc phỏng..."
4,Success Software Services,"Cty trẻ, năng động, lead có tâm, benefit ít.","Cty trẻ, năng động, lead có tâm. Process khá c...",Nên thêm các benefit tương ứng khi member làm ...
5,Success Software Services,Khách hàng chủ yếu thị trường Mỹ nên môi trườn...,Có thưởng dự án mỗi quý. Sếp luôn quan tâm đến...,Công ty cần tăng tỉ lệ đóng BHXH cho nhân viên...
6,Success Software Services,Hợp để fresher lấy kinh nghiệm.,"Môi trường cởi mở, đầu vào không quá khó, rất ...",Công ty cần tăng tỉ lệ đóng BHXH cho nhân viên...
7,Success Software Services,Môi trường phát triển tốt.,"Cơ hội giao tiếp bằng tiếng anh, làm việc với ...",Resource utilization phải làm nhiều project cù...
8,Success Software Services,"Môi trường năng động, phù hợp cho sinh viên mớ...",Môi trường năng động. Sếp thân thiện. Khá open...,Dự án gấp OT hơi nhiều. Lương thưởng phúc lợi ...
9,Success Software Services,Môi trường khá tốt.,Mô trường tương đối thân thiện. Quy trình ổn. ...,Văn phòng và cách sắp xếp bàn chưa thoải mái lắm.


In [ ]:
# Xóa các review trùng lặp (giữ lần đầu)
print("\nĐang xóa duplicate reviews...")
before_count = len(data)

# Xóa duplicate dựa trên review_content_masked
data_cleaned = data.drop_duplicates(subset=["review_content_masked"], keep="first")

# Loại bỏ các review rỗng sau khi làm sạch
data_cleaned = data_cleaned[data_cleaned["review_content_masked"].str.strip() != ""]

after_count = len(data_cleaned)
print(f"Đã xóa {before_count - after_count} reviews trùng lặp hoặc rỗng")
print(f"Số reviews còn lại: {after_count}")

data_cleaned.head(10)


Đang xóa duplicate...

Số dòng có giá trị sau khi xử lý:
  Title: 15436 dòng
  What I liked: 19857 dòng
  Suggestions for improvement: 17979 dòng


,Company Name,Cmt_day,Title,What I liked,Suggestions for improvement,Rating,Salary & benefits,Training & learning,Management cares about me,Culture & fun,Office & workspace,Recommend?
0,Success Software Services,2024-06-01,Phúc lợi khá OK còn công việc thì tùy dự án và...,"Văn phòng khá đẹp, nhiều event trong nội bộ cô...",Nên cải thiện quy trình và phúc lợi dành cho o...,3,3,3,3,3,3,Yes
1,Success Software Services,2023-09-01,"Môi trường năng động, trẻ, chính sách bảo mật ...","Môi trường năng động, trẻ trung. Có cơ hội làm...",Tăng số ngày nghỉ phép như 12 phép năm + 3 phé...,4,4,4,3,3,4,Yes
2,Success Software Services,2021-07-01,Một nơi tốt để học và phát triểu bản thân.,"Định hướng công ty tốt, khách hàng châu mỹ nên...","Cần đầu tư hơn về cơ sở vật chất, văn phòng , ...",5,4,5,5,4,3,Yes
3,Success Software Services,2024-03-01,"Review mang tính góp ý, không có ý định chê cô...","Đồng nghiệp thân thiện, hoà đồng, hỗ trợ hết m...","Ràng buộc thực tập sinh 2 năm, nhưng lúc phỏng...",4,2,2,4,4,3,No
4,Success Software Services,2024-02-01,"Cty trẻ, năng động, lead có tâm, benefit ít.","Cty trẻ, năng động, lead có tâm. Process khá c...",Nên thêm các benefit tương ứng khi member làm ...,3,2,3,3,2,4,No
5,Success Software Services,2021-07-01,Khách hàng chủ yếu thị trường Mỹ nên môi trườn...,Có thưởng dự án mỗi quý. Sếp luôn quan tâm đến...,Công ty cần tăng tỉ lệ đóng BHXH cho nhân viên...,5,5,5,5,5,4,Yes
6,Success Software Services,2019-02-01,Hợp để fresher lấy kinh nghiệm.,"Môi trường cởi mở, đầu vào không quá khó, rất ...",,4,3,4,3,4,3,Yes
7,Success Software Services,2018-11-01,Môi trường phát triển tốt.,"Cơ hội giao tiếp bằng tiếng anh, làm việc với ...",Resource utilization phải làm nhiều project cù...,3,3,3,2,2,3,Yes
8,Success Software Services,2017-06-01,"Môi trường năng động, phù hợp cho sinh viên mớ...",Môi trường năng động - Sếp thân thiện - Khá op...,Dự án gấp OT hơi nhiều - Lương thưởng phúc lợi...,4,3,4,3,4,3,Yes
9,Success Software Services,2016-11-01,Môi trường khá tốt.,Mô trường tương đối thân thiện Quy trình ổn Tạ...,Văn phòng và cách sắp xếp bàn chưa thoải mái lắm.,4,3,4,4,4,3,Yes


In [ ]:
# Xuất file kết quả
data_cleaned.to_csv("reviews_cleaned.csv", index=False, encoding='utf-8-sig')
print("Đã xuất file reviews_cleaned.csv")
print(f"Tổng số reviews: {len(data_cleaned)}")
print(f"Số công ty: {data_cleaned['company_name'].nunique()}")

Đã xuất file Vietnamese_reviews_cleaned.xlsx


In [ ]:
# Lưu file CSV cho 100 dòng đầu để labeling
data_100 = data_cleaned.head(100).copy()

# Chọn các cột cần thiết
data_100_export = data_100[["reviewer_id", "company_name", "review_content_masked"]].copy()

# Lưu file để chuẩn bị gán nhãn
data_100_export.to_csv("reviews_100_for_labeling.csv", index=False, encoding='utf-8-sig')
print("Đã lưu file reviews_100_for_labeling.csv (100 dòng đầu, sẵn sàng để gán nhãn)")
print(f"\nMột số mẫu:")
data_100_export.head(10)

Đã lưu file Vietnamese_100_cleaned_masked.csv (100 dòng đầu, đã mask tên công ty thành [COMPANY])


In [ ]:
# Tạo thống kê theo công ty
print("=== THỐNG KÊ THEO CÔNG TY ===\n")
company_stats = data_cleaned.groupby('company_name').agg({
    'reviewer_id': 'count',
    'review_content_masked': lambda x: x.str.len().mean()
}).round(0)
company_stats.columns = ['Số reviews', 'Độ dài TB (ký tự)']
company_stats = company_stats.sort_values('Số reviews', ascending=False)

print("Top 20 công ty có nhiều reviews nhất:")
print(company_stats.head(20))

# Lưu thống kê
company_stats.to_csv("company_statistics.csv", encoding='utf-8-sig')
print("\nĐã lưu thống kê vào company_statistics.csv")

Không tìm thấy file Vietnamese_100_cleaned_labeled.csv


In [ ]:
# Phân tích độ dài reviews
import matplotlib.pyplot as plt
import numpy as np

review_lengths = data_cleaned['review_content_masked'].str.len()

print("=== PHÂN TÍCH ĐỘ DÀI REVIEWS ===")
print(f"Độ dài trung bình: {review_lengths.mean():.0f} ký tự")
print(f"Độ dài trung vị: {review_lengths.median():.0f} ký tự")
print(f"Độ dài ngắn nhất: {review_lengths.min():.0f} ký tự")
print(f"Độ dài dài nhất: {review_lengths.max():.0f} ký tự")
print(f"Độ lệch chuẩn: {review_lengths.std():.0f} ký tự")

# Vẽ histogram
plt.figure(figsize=(12, 6))
plt.hist(review_lengths, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Độ dài review (ký tự)')
plt.ylabel('Số lượng')
plt.title('Phân bố độ dài reviews')
plt.axvline(review_lengths.mean(), color='red', linestyle='--', label=f'Trung bình: {review_lengths.mean():.0f}')
plt.axvline(review_lengths.median(), color='green', linestyle='--', label=f'Trung vị: {review_lengths.median():.0f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nPhân vị 25%: {review_lengths.quantile(0.25):.0f} ký tự")
print(f"Phân vị 75%: {review_lengths.quantile(0.75):.0f} ký tự")

Checking for label columns...
Đã tạo file Vietnamese_100_cleaned_labeled_masked.csv với Labels từ rating columns

Số dòng có Labels: 100/100


,Combined_Review,Labels
0,Phúc lợi khá OK còn công việc thì tùy dự án và...,"Salary & benefits, Training & learning, Manage..."
1,"Môi trường năng động, trẻ, chính sách bảo mật ...","Salary & benefits, Training & learning, Manage..."
2,Một nơi tốt để học và phát triểu bản thân. Địn...,"Salary & benefits, Training & learning, Manage..."
3,"Review mang tính góp ý, không có ý định chê cô...","Salary & benefits, Training & learning, Manage..."
4,"Cty trẻ, năng động, lead có tâm, benefit ít. C...","Salary & benefits, Training & learning, Manage..."
5,Khách hàng chủ yếu thị trường Mỹ nên môi trườn...,"Salary & benefits, Training & learning, Manage..."
6,Hợp để fresher lấy kinh nghiệm. Môi trường cởi...,"Salary & benefits, Training & learning, Manage..."
7,Môi trường phát triển tốt. Cơ hội giao tiếp bằ...,"Salary & benefits, Training & learning, Manage..."
8,"Môi trường năng động, phù hợp cho sinh viên mớ...","Salary & benefits, Training & learning, Manage..."
9,Môi trường khá tốt. Mô trường tương đối thân t...,"Salary & benefits, Training & learning, Manage..."


In [ ]:
# Kiểm tra và hiển thị một số review mẫu
print("=== MỘT SỐ REVIEW MẪU SAU KHI XỬ LÝ ===\n")

sample_reviews = data_cleaned.sample(min(5, len(data_cleaned)), random_state=42)

for idx, row in sample_reviews.iterrows():
    print(f"Công ty: {row['company_name']}")
    print(f"Review: {row['review_content_masked'][:200]}...")
    print(f"Độ dài: {len(row['review_content_masked'])} ký tự")
    print("-" * 80)
    print()

Đã gộp 3 cột thành 1 cột 'Combined_Review' cho 19938 dòng đầu

Độ dài trung bình: 385 ký tự


,Combined_Review
0,Phúc lợi khá OK còn công việc thì tùy dự án và...
1,"Môi trường năng động, trẻ, chính sách bảo mật ..."
2,Một nơi tốt để học và phát triểu bản thân. Địn...
3,"Review mang tính góp ý, không có ý định chê cô..."
4,"Cty trẻ, năng động, lead có tâm, benefit ít. C..."
5,Khách hàng chủ yếu thị trường Mỹ nên môi trườn...
6,Hợp để fresher lấy kinh nghiệm. Môi trường cởi...
7,Môi trường phát triển tốt. Cơ hội giao tiếp bằ...
8,"Môi trường năng động, phù hợp cho sinh viên mớ..."
9,Môi trường khá tốt. Mô trường tương đối thân t...


In [ ]:
# Phân tích từ tiếng Anh trong reviews
import re
from collections import Counter
import pandas as pd
import nltk
nltk.download('words', quiet=True)
from nltk.corpus import words

english_vocab = set(words.words())

# Blacklist: từ tiếng Việt không dấu trùng với tiếng Anh
VIETNAMESE_BLACKLIST = {
    'co', 'con', 'ban', 'chi', 'cho', 'chua', 'chu', 'cua', 'da', 'dan',
    'dao', 'den', 'di', 'do', 'doi', 'du', 'duoc', 'giao', 'gio', 'hai',
    'hay', 'het', 'ho', 'hoc', 'khi', 'khong', 'la', 'lam', 'lon', 'ma',
    'mai', 'mat', 'met', 'moi', 'nam', 'nao', 'nay', 'nha', 'nhi', 'nho',
    'nhu', 'nhung', 'noi', 'qua', 'quan', 'rat', 'roi', 'sau', 'se', 'tai',
    'tam', 'than', 'the', 'thi', 'thoi', 'thu', 'tot', 'trang', 'tre', 'tren',
    'trong', 'tu', 'va', 've', 'vi', 'voi', 'xa', 'xin', 'xu', 'ca', 'cac',
    'anh', 'chi', 'em', 'ong', 'ba', 'de', 'dong', 'long', 'tin', 'tri', 'sinh',
}

try:
    import enchant
    d = enchant.Dict("en_US")
    
    def extract_english_words_enchant(text):
        words_found = re.findall(r'\b[a-zA-Z]{4,}\b', str(text).lower())
        return [w for w in words_found if d.check(w) and w not in VIETNAMESE_BLACKLIST]
    
    # Tổng hợp tất cả từ tiếng Anh
    all_words = []
    for review in data_cleaned["review_content_masked"]:
        all_words.extend(extract_english_words_enchant(review))
    
    # Thống kê tần suất
    word_counts = Counter(all_words)
    print(f"\n=== PHÂN TÍCH TỪ TIẾNG ANH ===")
    print(f"Tổng số từ tiếng Anh tìm thấy: {len(word_counts)} từ khác nhau")
    print(f"\nTop 30 từ xuất hiện nhiều nhất:")
    for word, count in word_counts.most_common(30):
        print(f"  {word}: {count}")
        
except ImportError:
    print("Cần cài đặt thư viện enchant để phân tích từ tiếng Anh")
    print("Chạy: pip install pyenchant")

False